# Section 2: Multivariate assimilation

*(Replaces DART_LAB slide deck Section 2.)*

Real models have many state variables but few are observed directly. The
key idea of ensemble assimilation: the *joint prior ensemble* tells us how
an observed variable relates to every unobserved one, so observation
increments can be **regressed** onto each state variable independently.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pydartlab as dl
import pydartlab.apps as apps

## Updating an unobserved variable

For each ensemble member $n$, the increment of an unobserved state variable
$x$ is a linear function of the observed variable $y$'s increment:

$$ \Delta x_n = \frac{\widehat{\mathrm{cov}}(x, y)}{\widehat{\mathrm{var}}(y)}\, \Delta y_n $$

— a least-squares regression using the prior ensemble's sample covariance.
This is done for every state variable, one at a time, and observations
with independent errors can be assimilated **sequentially** (one after the
other), which is exactly what DART does.

## Exercise: `twod_ensemble`

The horizontal axis is observed; the vertical axis is not. Create joint
ensembles by clicking in the central panel and **Update**:

1. Members nearly on a line (strong correlation): the unobserved variable
   is updated almost as strongly as the observed one.
2. A round, uncorrelated cloud: the unobserved variable barely moves.
3. A bimodal joint distribution, and a cloud with one outlier — watch how
   the regression (a straight line!) handles them.

In [ ]:
te = apps.twod_ensemble()
te

In [ ]:
# Mouse-free, e.g. the correlated case:
te.set_ensemble([(2, 2.5), (3, 3.2), (4, 4.4), (5, 5.1), (6, 6.3), (7, 6.8)])
te.update_ensemble()

## From two variables to a real model

Nothing changes with more variables: observe what you can, regress the
increments onto everything. The next two tools apply exactly the
1D filters + regression of Sections 1-2 to chaotic models:

* **Lorenz 63** — the 3-variable "butterfly" attractor,
* **Lorenz 96** — a 40-variable cyclic analogue of midlatitude flow:
  $\dot x_j = (x_{j+1} - x_{j-2})\,x_{j-1} - x_j + F$.

## Exercise: `run_lorenz_63`

1. Leave the filter on **No Assimilation** and auto-run: the 20 green
   ensemble members spread out over the attractor — chaos at work.
2. Reset, choose **EAKF**, and step through advance/assimilate cycles:
   red segments show the increments pulling members back to the truth.
3. Pause and rotate the 3-D axes. Where on the attractor do the ensemble
   members spread fastest? (Hint: near the saddle between the lobes.)

In [ ]:
l63 = apps.run_lorenz_63(seed=4)
l63

## Exercise: `run_lorenz_96`

1. **Free run** (No Assimilation): tiny initial perturbations take ~20
   steps to grow (spin-up), then the error saturates — remember this
   saturation level; it's what "no skill" looks like.
2. Switch to **EAKF** and run: the error drops well below saturation.
3. Compare prior and posterior rank histograms.

(Keep localization = 1.0 and inflation = 1.0 for now — they're the subject
of Section 3.)

In [ ]:
l96 = apps.run_lorenz_96(seed=4)
l96

In [ ]:
# Scripted: free run vs EAKF, 150 cycles
from pydartlab.experiments import Lorenz96Experiment

fig, ax = plt.subplots(figsize=(7, 3.2))
for name, kwargs in {
    "free run": dict(filter_type="No Assimilation"),
    "EAKF": dict(filter_type="EAKF", localization=0.2),
}.items():
    exp = Lorenz96Experiment(seed=11, **kwargs)
    for _ in range(150):
        exp.step()
    ax.plot(exp.history["time"], exp.history["prior_error"], label=name)
ax.set_xlabel("time step"); ax.set_ylabel("prior RMS error")
ax.legend(); ax.set_title("Assimilation vs error saturation");

## What you should have seen

* Correlation in the joint prior is what lets observations update
  unobserved variables; no correlation, no update.
* The same 1D filters + regression assimilate a chaotic 3- and 40-variable
  model.
* A free-running ensemble saturates at climatological error; assimilation
  holds the error far below that.

**Next: Section 3 — two things that go wrong (too little spread, spurious
correlations) and their fixes: inflation and localization.**